# دبلجة Billionaire Granny إلى العربية — تشغيل فعلي على Google Colab

هذا الدفتر يستخدم الحلقة التي شاركتها، ويعيد استخدام ملف التوقيت السابق ذي **864 مقطعًا و4 مجموعات متحدثين**. لا يعيد تشغيل Whisper من البداية. يعتمد على Edge-TTS بأصوات عربية اصطناعية مختلفة، ويحتفظ بنسخ الاستئناف في Google Drive.

**المصدر:** [فتح الحلقة في Drive](https://drive.google.com/file/d/1sMis6qi_j3AsBvAEL2mliPRIKf0s3-uI/view)

**المخرجات:** مجلد [Netflix_Arabic_Dubs في Drive](https://drive.google.com/drive/folders/1XuvZ31iemwjVDZ4TgQYmrJVGWAh3VqJF)

### ما الذي يلزم؟
1. شغّل الخلية الأولى ووافق على ربط Google Drive بالحساب الذي يحتوي ملف البيانات السابق.
2. شغّل الخلايا بالترتيب أو اختر **Runtime → Run all**.
3. اترك خلية الدبلجة تعمل. سيُحفظ التقدم والترجمات ومقاطع الصوت في Drive كلما اكتملت دفعات.

### حماية الذاكرة والاستئناف
الدفتر يفحص الذاكرة ومساحة القرص قبل تنزيل الحلقة أو النماذج. يحتاج إلى 10 GiB على الأقل من RAM إجمالًا، و8 GiB متاحة، و14 GiB مساحة محلية فارغة. إذا لم يتحقق ذلك، يتوقف قبل المعالجة المكلفة. يجري تركيب الصوت وخلطه على القرص في كتل صغيرة بدل تحميل مسار ساعتين كاملًا إلى الذاكرة.

**حدود الجودة:** سيتولد مسار صوتي عربي متعدد الشخصيات مع توقيت وحِدّة صوتية تقريبية بحسب نوع الحوار. الخط الحالي لا يتتبّع الوجوه ولا يحرّك الفم داخل الصورة؛ لذلك سيُسجّل تقرير الجودة أن **مزامنة الشفاه البصرية لم تُنفّذ**. كما أن الترجمة الآلية تحتاج مراجعة لغوية وبشرية قبل وصف النتيجة بأنها بمستوى نتفليكس.

In [ ]:
from google.colab import drive

# قد تظهر نافذة Google للموافقة على الوصول إلى Drive.
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil
import json
import os

DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_FOLDER = DRIVE_ROOT / 'Netflix_Arabic_Dubs'
METADATA_FILE = DRIVE_FOLDER / 'Billionaire_Granny_CEO_Dubbing_Metadata_864_Segments.json'
SOURCE_URL = 'https://drive.google.com/file/d/1sMis6qi_j3AsBvAEL2mliPRIKf0s3-uI/view?usp=drivesdk'

if not DRIVE_ROOT.is_dir():
    raise RuntimeError("لم يتم ربط Google Drive. أعد تشغيل خلية ربط Drive أولاً.")
if not METADATA_FILE.is_file():
    raise FileNotFoundError(
        "ملف التوقيت السابق غير موجود: " + str(METADATA_FILE) +
        "\nيجب أن يوجد ملف Billionaire_Granny_CEO_Dubbing_Metadata_864_Segments.json داخل Netflix_Arabic_Dubs."
    )

memory = {}
for line in Path('/proc/meminfo').read_text(encoding='utf-8').splitlines():
    key, _, value = line.partition(':')
    if value.strip().endswith('kB'):
        memory[key] = int(value.strip().split()[0]) / (1024 * 1024)

ram_total = memory.get('MemTotal', 0.0)
ram_available = memory.get('MemAvailable', 0.0)
disk_free_gib = shutil.disk_usage('/content').free / (1024**3)

print("ملف التوقيت:", METADATA_FILE)
print(f"RAM الإجمالي: {ram_total:.2f} GiB")
print(f"RAM المتاح: {ram_available:.2f} GiB")
print(f"مساحة القرص المحلي المتاحة: {disk_free_gib:.2f} GiB")
print("مجلد المخرجات:", DRIVE_FOLDER)

if ram_total < 10:
    raise RuntimeError(f"الذاكرة الإجمالية {ram_total:.2f} GiB أقل من الحد الآمن 10 GiB. لم يبدأ تنزيل الحلقة أو النماذج.")
if ram_available < 8:
    raise RuntimeError(f"المتاح من RAM هو {ram_available:.2f} GiB فقط. أعد تشغيل بيئة Colab ثم أعد الاختبار.")
if disk_free_gib < 14:
    raise RuntimeError(f"مساحة القرص المحلي المتاحة {disk_free_gib:.2f} GiB فقط. يلزم 14 GiB على الأقل.")

DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)
write_test = DRIVE_FOLDER / '.colab_write_test.tmp'
write_test.write_text('ok', encoding='utf-8')
write_test.unlink()
print("فحص الذاكرة والقرص والكتابة إلى Drive: ناجح.")

In [ ]:
import subprocess
import shutil
from pathlib import Path

REPO = Path('/content/AI-Creative-Studio')
SCRIPT = REPO / 'scripts' / 'dub_billionaire_granny_edge_tts.py'

def run_setup(command, label):
    print("\\n== " + label + " ==", flush=True)
    process = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    if process.stdout:
        print(process.stdout[-10000:], flush=True)
    if process.returncode:
        raise RuntimeError(f"{label} فشل برمز {process.returncode}.")

if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    run_setup(['apt-get', 'update'], 'تحديث قائمة الحزم')
    run_setup(['apt-get', 'install', '-y', 'ffmpeg', 'git'], 'تثبيت FFmpeg وGit')

if not REPO.is_dir():
    run_setup(['git', 'clone', '--depth', '1', 'https://github.com/h776117073-eng/AI-Creative-Studio.git', str(REPO)],
              'تنزيل مستودع AI-Creative-Studio')
else:
    run_setup(['git', '-C', str(REPO), 'pull', '--ff-only'], 'تحديث مستودع AI-Creative-Studio')

if not SCRIPT.is_file():
    raise FileNotFoundError(f"سكربت الدبلجة غير موجود: {SCRIPT}")

print("\\nالبيئة جاهزة. لن يتم تحميل الفيديو كاملاً إلى RAM؛ المعالجة والخلط يستخدمان ملفات على القرص.")

## بدء الدبلجة
تبدأ هذه الخلية بتنزيل الحلقة إلى قرص Colab المحلي، ثم ترجمة المقاطع وتوليد الأصوات. سيبقى السجل ظاهرًا في الخرج، ويُحفظ ملف التقدم ودفعات الأصوات في Drive. عند انقطاع الجلسة، أعد ربط Drive ثم أعد تشغيل الخلية؛ ستُستعاد دفعات الصوت المحفوظة.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/AI-Creative-Studio')
SCRIPT = REPO / 'scripts' / 'dub_billionaire_granny_edge_tts.py'
METADATA_FILE = Path('/content/drive/MyDrive/Netflix_Arabic_Dubs/Billionaire_Granny_CEO_Dubbing_Metadata_864_Segments.json')

os.environ['METADATA_FILE'] = str(METADATA_FILE)
os.environ['SOURCE_URL'] = 'https://drive.google.com/file/d/1sMis6qi_j3AsBvAEL2mliPRIKf0s3-uI/view?usp=drivesdk'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'

if not SCRIPT.is_file():
    raise FileNotFoundError(SCRIPT)
if not METADATA_FILE.is_file():
    raise FileNotFoundError(METADATA_FILE)

print("بدأت مهمة الدبلجة. لا توقف الجلسة أثناء المعالجة.", flush=True)
print("إذا انقطعت الجلسة، أعد ربط Drive وشغّل هذه الخلية مجددًا لاستعادة المقاطع المؤكدة من checkpoints.", flush=True)

with subprocess.Popen(
    [sys.executable, str(SCRIPT)],
    cwd=str(REPO),
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors='replace',
    bufsize=1,
) as process:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()

if code != 0:
    progress = Path('/content/drive/MyDrive/Netflix_Arabic_Dubs/Billionaire_Granny_CEO_Progress.json')
    print("\\nآخر حالة محفوظة:", flush=True)
    if progress.is_file():
        print(progress.read_text(encoding='utf-8'), flush=True)
    raise RuntimeError(
        f"توقفت عملية الدبلجة برمز {code}. افتح ملف Billionaire_Granny_CEO_Progress.json "
        "وسجل Billionaire_Granny_CEO_Progress.log داخل Drive لمعرفة المرحلة، ثم أعد تشغيل الخلية."
    )

print("\\nانتهى السكربت بنجاح. افحص تقرير الجودة في الخلية التالية.", flush=True)

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown

OUT = Path('/content/drive/MyDrive/Netflix_Arabic_Dubs')
FINAL = OUT / 'Billionaire_Granny_CEO_Full_Arabic_Dub.mp4'
REPORT = OUT / 'Billionaire_Granny_CEO_Quality_Report.json'
PROGRESS = OUT / 'Billionaire_Granny_CEO_Progress.json'

if PROGRESS.is_file():
    status = json.loads(PROGRESS.read_text(encoding='utf-8'))
    print("آخر مرحلة:", status.get('stage', status.get('status')))
    print("آخر تحديث UTC:", status.get('updated_at_utc'))

if REPORT.is_file():
    report = json.loads(REPORT.read_text(encoding='utf-8'))
    print(json.dumps({
        "status": report.get("status"),
        "source_duration_seconds": report.get("source_duration_seconds"),
        "final_duration_seconds": report.get("final_duration_seconds"),
        "speaker_count": report.get("speaker_count"),
        "segment_count": report.get("segment_count"),
        "arabic_text_coverage": report.get("arabic_text_coverage"),
        "generated_tts_segments": report.get("generated_tts_segments"),
        "final_size_bytes": report.get("final_size_bytes"),
        "final_sha256": report.get("final_sha256"),
        "ffprobe_stream_check": report.get("ffprobe_stream_check"),
        "full_ffmpeg_decode": report.get("full_ffmpeg_decode"),
        "drive_copy_checksum": report.get("drive_copy_checksum"),
        "visual_lip_sync": report.get("visual_lip_sync"),
    }, ensure_ascii=False, indent=2))
else:
    print("لا يوجد تقرير نهائي متحقق بعد.")
    if FINAL.is_file():
        print("يوجد ملف MP4، لكن لا تعتبره نهائيًا قبل ظهور quality_report.json وفحوص التحقق.")
    else:
        print("الفيديو النهائي لم يُنشأ بعد.")

print("\\nمجلد النتائج في Drive:")
print("https://drive.google.com/drive/folders/1XuvZ31iemwjVDZ4TgQYmrJVGWAh3VqJF")
